# 🟢 Notebook 4 — Agente de Análisis de Datos con Herramientas
> **LLM que llama a modelos ML como tools**

En este notebook vamos a:
1. Construir un agente con LangChain + Gemini que tiene acceso a herramientas de análisis
2. El agente puede llamar funciones de ML, estadísticas y visualización
3. El usuario hace preguntas en lenguaje natural y el agente orquesta el análisis

**Módulo:** IA Generativa — Clase 4: Pipelines ML + GenAI

## 1. Instalación de dependencias

In [ ]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

## 2. Configuración

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

print("✅ Configuración lista")

## 3. Dataset de ventas globales

In [ ]:
np.random.seed(42)
n = 200

DATASET = pd.DataFrame({
    "producto": np.random.choice(["Laptop", "Tablet", "Auriculares", "Teclado", "Monitor"], n),
    "region": np.random.choice(["Norte", "Sur", "Este", "Oeste"], n),
    "mes": np.random.randint(1, 13, n),
    "ventas_unidades": np.random.randint(10, 500, n),
    "precio_unitario": np.random.uniform(20, 800, n).round(2),
    "satisfaccion_cliente": np.random.uniform(2.0, 5.0, n).round(1),
    "descuento_pct": np.random.uniform(0, 0.3, n).round(2),
})

DATASET["revenue"] = (DATASET["ventas_unidades"] * 
                       DATASET["precio_unitario"] * 
                       (1 - DATASET["descuento_pct"])).round(2)

print(f"Dataset de ventas cargado: {DATASET.shape}")
DATASET.head()

## 4. Definir las herramientas del agente

In [ ]:
@tool
def obtener_estadisticas(columna: str) -> str:
    """
    Obtiene estadísticas descriptivas de una columna del dataset de ventas.
    Columnas disponibles: producto, region, mes, ventas_unidades, precio_unitario,
    satisfaccion_cliente, descuento_pct, revenue
    """
    if columna not in DATASET.columns:
        return f"Error: columna '{columna}' no existe. Disponibles: {list(DATASET.columns)}"
    
    if DATASET[columna].dtype in ["object"]:
        stats = DATASET[columna].value_counts().to_dict()
        return f"Distribución de '{columna}': {json.dumps(stats, ensure_ascii=False)}"
    else:
        stats = DATASET[columna].describe().round(2).to_dict()
        return f"Estadísticas de '{columna}': {json.dumps(stats)}"


@tool  
def top_productos_por_revenue(n: int = 5) -> str:
    """
    Retorna los N productos con mayor revenue total en el dataset.
    Parámetro n: número de productos a mostrar (default 5).
    """
    top = (DATASET.groupby("producto")["revenue"]
           .sum()
           .sort_values(ascending=False)
           .head(n)
           .round(2)
           .to_dict())
    return f"Top {n} productos por revenue: {json.dumps(top)}"


@tool
def segmentar_clientes_kmeans(n_clusters: int = 3) -> str:
    """
    Aplica K-Means clustering sobre precio_unitario, ventas_unidades y satisfaccion_cliente.
    Retorna el perfil de cada segmento. n_clusters debe ser entre 2 y 5.
    """
    features = ["precio_unitario", "ventas_unidades", "satisfaccion_cliente", "descuento_pct"]
    X = DATASET[features].copy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    DATASET["segmento"] = km.fit_predict(X_scaled)
    
    perfil = DATASET.groupby("segmento")[features + ["revenue"]].mean().round(2)
    sizes = DATASET["segmento"].value_counts().to_dict()
    
    result = {}
    for seg in range(n_clusters):
        result[f"Segmento_{seg}"] = {
            **perfil.loc[seg].to_dict(),
            "tamaño": sizes[seg]
        }
    return f"Segmentación en {n_clusters} clusters: {json.dumps(result)}"


@tool
def predecir_revenue(precio: float, unidades: int, descuento: float) -> str:
    """
    Predice el revenue esperado para un escenario dado usando regresión lineal.
    precio: precio unitario del producto
    unidades: cantidad de unidades a vender
    descuento: porcentaje de descuento (entre 0 y 1, ej: 0.15 = 15%)
    """
    X = DATASET[["precio_unitario", "ventas_unidades", "descuento_pct"]]
    y = DATASET["revenue"]
    
    modelo = LinearRegression()
    modelo.fit(X, y)
    r2 = r2_score(y, modelo.predict(X))
    
    pred = modelo.predict([[precio, unidades, descuento]])[0]
    
    return (f"Revenue predicho: ${pred:,.2f} "
            f"(R² del modelo: {r2:.3f}) "
            f"para precio=${precio}, unidades={unidades}, descuento={descuento:.0%}")


@tool
def comparar_regiones(metrica: str = "revenue") -> str:
    """
    Compara el rendimiento de las 4 regiones (Norte, Sur, Este, Oeste) 
    para una métrica dada.
    Métricas válidas: revenue, ventas_unidades, satisfaccion_cliente, descuento_pct
    """
    if metrica not in ["revenue", "ventas_unidades", "satisfaccion_cliente", "descuento_pct"]:
        return f"Métrica '{metrica}' no válida."
    
    comparacion = (DATASET.groupby("region")[metrica]
                   .agg(["mean", "sum", "count"])
                   .round(2)
                   .to_dict())
    return f"Comparación por región ({metrica}): {json.dumps(comparacion)}"


tools = [obtener_estadisticas, top_productos_por_revenue, 
         segmentar_clientes_kmeans, predecir_revenue, comparar_regiones]

print(f"✅ {len(tools)} herramientas definidas:")
for t in tools:
    print(f"  • {t.name}")

## 5. Construir el agente con LangChain

In [ ]:
system_prompt = """
Eres DataBot, un analista de datos experto. Tienes acceso a herramientas para analizar 
un dataset de ventas que incluye: productos (Laptop, Tablet, Auriculares, Teclado, Monitor),
regiones (Norte, Sur, Este, Oeste), métricas de ventas, precios y satisfacción del cliente.

Cuando el usuario haga preguntas:
1. Usa las herramientas apropiadas para obtener datos reales
2. Interpreta los resultados con contexto de negocio
3. Da recomendaciones accionables
4. Si necesitas múltiples herramientas, úsalas en secuencia
5. Responde siempre en español
"""

prompt_agente = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agente = create_tool_calling_agent(llm, tools, prompt_agente)
ejecutor = AgentExecutor(
    agent=agente,
    tools=tools,
    verbose=True,
    max_iterations=5
)

print("✅ Agente creado y listo")

## 6. Interacción con el agente

In [ ]:
def preguntar(pregunta: str):
    print(f"\n{'='*60}")
    print(f"🧑 USUARIO: {pregunta}")
    print("="*60)
    result = ejecutor.invoke({"input": pregunta})
    print(f"\n🤖 DATABOT: {result['output']}")
    return result

# Pregunta 1: análisis simple
preguntar("¿Cuál es el revenue promedio y el producto que más vende?")

In [ ]:
# Pregunta 2: que requiere múltiples herramientas
preguntar("Compara el rendimiento de las regiones por revenue y dime cuál tiene mejor satisfacción del cliente")

In [ ]:
# Pregunta 3: análisis avanzado con clustering
preguntar("Segmenta las ventas en 3 grupos y explícame el perfil de cada segmento")

In [ ]:
# Pregunta 4: predicción
preguntar("Si vendo Laptops a $600 con un 10% de descuento y espero vender 100 unidades, ¿cuánto revenue generaría?")

## 7. Modo interactivo (para ejecutar en Jupyter)

In [ ]:
# Descomenta para modo chat interactivo
# print("DataBot listo! (escribe 'salir' para terminar)")
# while True:
#     pregunta = input("\nTú: ")
#     if pregunta.lower() in ["salir", "exit", "quit"]:
#         print("¡Hasta luego!")
#         break
#     preguntar(pregunta)

## 🎯 Conclusiones

- Los agentes LangChain pueden **orquestar múltiples modelos y funciones** en respuesta a lenguaje natural
- El agente decide **qué herramientas usar y en qué orden** según la pregunta
- Este patrón permite crear **interfaces conversacionales** sobre datos sin que el usuario sepa ML

**Patrón clave:** `Pregunta natural → Agente → Tool calling (ML, stats) → Síntesis en lenguaje natural`